In [ ]:
!pip install "pandas<2.2.0" wfdb

In [ ]:
import wfdb
import numpy as np
import pandas as pd
import math
from google.colab import files # Librería para descargar en Colab
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

#CSV

In [ ]:
# ---------- Parámetros ----------
THRESHOLD = 0.3
WAIT_SAMPLES = 72
FS = 360
REFRACT_SAMPLES = 72
K = 4

# ---------- Parámetros Hermites ----------
SIGMA = 5.0
NUM_COEFFS = 5
WINDOW = 36 # Mitad del latido (72/2)

# ---------- Buffers ----------
TAM_BUF = 150 #dejamos margen

In [ ]:
# --- 1. FUNCIÓN HERMITE ---
def hermite(t, n, sigma):
    x = t / sigma

    # --- 1. CALCULO DEL POLINOMIO Hn(x) ---
    # H_n = 2x*H_{n-1} - 2(n-1)*H_{n-2}

    if n == 0:
        Hn = 1.0
    elif n == 1:
        Hn = 2.0 * x
    else:
        h_n_2 = 1.0       # H0
        h_n_1 = 2.0 * x        # H1
        Hn = 0.0

        # Bucle desde 2 hasta n
        for i in range(2, n + 1):
            # Aplicamos la fórmula
            Hn = 2.0 * x * h_n_1 - 2.0 * (i - 1) * h_n_2

            # Actualizo para la siguiente vuelta
            h_n_2 = h_n_1
            h_n_1 = Hn

    # --- 2. Gaussiana ---
    gauss = math.exp(- (t**2) / (2 * sigma**2))

    # --- 3. Constante normalizacion --
    K = 1.0 / math.sqrt(sigma * (2**n) * math.factorial(n) * math.sqrt(math.pi))

    return K * gauss * Hn

# --- 2. FUNCIÓN PROCESAR LATIDO ---
def procesar_latido(fragmento_latido):
  # fragmento_latido es el array de 72 muestras
  # Copiamos
  datos_analizar = list(fragmento_latido) # (C -> memcpy o for)

  # Calculamos media
  suma = sum(datos_analizar)
  media = suma / len(datos_analizar)

  # Restamos media (centrar en 0)
  for j in range(len(datos_analizar)):
    datos_analizar[j] -= media

  # CALCULO COEFS
  mis_coefs = []
  for n in range(NUM_COEFFS):
    result = 0.0

    for j in range(len(datos_analizar)):
      t = j - WINDOW
      c_n = datos_analizar[j]
      phi_n = hermite(t, n, SIGMA)
      result += c_n * phi_n
    mis_coefs.append(result)

  return mis_coefs

In [ ]:
# Lista de registros limpios
registros_limpios = [
    '100', '101', '102', '103', '104', '105', '106', '109', '113', '114',
    '122', '123', '200', '202', '205', '207', '209', '210', '214', '215',
    '220', '221', '222', '230', '231', '232'
]

dataset_filas = []
print("Empezando a procesar registros...")

for num_registro in registros_limpios:
    print(f" -> Leyendo registro {num_registro}")

    # ---------- Leer señal y Anotaciones ----------
    record = wfdb.rdrecord(num_registro, pn_dir='mitdb')
    signal = record.p_signal[:,0]   # canal 0
    N = len(signal)

    # Leer el archivo .atr
    anotacion = wfdb.rdann(num_registro, 'atr', pn_dir='mitdb')
    posiciones_reales = anotacion.sample
    tipos_reales = anotacion.symbol

    # ---------- Inicializar variables  ----------
    state = "RESET"
    last_confirmed = -100000
    buf_latido = [0.0] * TAM_BUF

    cand_val = -1.0
    cand_idx = -1
    prov_best_val = -1.0
    prov_best_idx = -1
    wait_until = -1
    umbral_descendiente = 0.0

    # ---------- Bucle principal ----------
    for i in range(K, N):

        # 0) BUFFER HERMITE
        buf_latido.pop(0) #borro el viejo
        buf_latido.append(signal[i]) #añado el nuevo


        # 1) derivada simple
        d = signal[i] - signal[i-K]
        v = abs(d)   # valor absoluto
        idx = i-4

        umbral_descendiente = umbral_descendiente * 0.99


        # ---------- Máquina de estados ----------
        if state == "RESET":
            # reiniciar
            state = "LOOKING"
            cand_val = -1.0 # valor candidato a QRS
            cand_idx = -1 # pos del candidato a QRS
            prov_best_idx = -1 # valor def QRS
            prov_best_val = -1.0 #pos de def QRS
            wait_until = -1


        if state == "LOOKING":
            if v > THRESHOLD and v > umbral_descendiente: #si hay subida notable
                state = "PROV"
                cand_val = v
                cand_idx = idx


        elif state == "PROV":
            if v > cand_val: #si val > max anterior
                cand_val = v
                cand_idx = idx
            # si baja a menos de la mitad del pico provisional -> pasamos a HALF
            if v < 0.5 * cand_val:
                state = "HALF"
                prov_best_val = cand_val
                prov_best_idx = cand_idx
                wait_until = prov_best_idx + WAIT_SAMPLES #ESPERAMOS 72 samples


        elif state == "HALF":
            if v > prov_best_val: #si aparece v mayor que max hasta ahora
                state = "PROV" #volvemos a PROV
                cand_val = v
                cand_idx = idx
            else:
                # si esperamos suficiente, confirmamos el pico
                if i >= wait_until:
                    if prov_best_idx - last_confirmed > REFRACT_SAMPLES:
                    #si no cumple el REFRACTORIO no se confirma pico
                      if prov_best_val > umbral_descendiente:
                          #detected.append(int(prov_best_idx))
                          last_confirmed = prov_best_idx #pos del ultimo pico detectado
                          umbral_descendiente = prov_best_val

                          # Calcular donde está el pico en el buf
                          lag = i - prov_best_idx
                          idx_buf = (TAM_BUF - 1) - lag

                          # Calcular ventana 72 muestras
                          inicio = idx_buf - WINDOW
                          fin = idx_buf + WINDOW

                          # Comprobamos que no nos salimos
                          if inicio >= 0 and fin <= TAM_BUF:
                              # Cortamos el trozo exacto de 72 muestras
                              recorte = buf_latido[inicio:fin]

                              # --- LLAMADA FUNCIÓN ---
                              coeficientes = procesar_latido(recorte)

                              # --- BUSCAR QUÉ DICE EL CARDIÓLOGO ---
                              # Miramos qué latido de la base de datos oficial está más cerca del nuestro
                              distancias = np.abs(posiciones_reales - prov_best_idx)
                              idx_mas_cercano = np.argmin(distancias)

                              # Si el médico marcó algo a menos de 20 muestras de nuestro pico, adjudico etiqueta
                              if distancias[idx_mas_cercano] <= 20:
                                  etiqueta = tipos_reales[idx_mas_cercano]
                              else:
                                  etiqueta = "No_Anotado"

                              # Guardamr
                              fila = {
                                  'Registro': num_registro,
                                  'Muestra_Pico': prov_best_idx,
                                  'Clase_Cardiologo': etiqueta
                              }
                              # coefs
                              for num_c in range(NUM_COEFFS):
                                  fila[f'c{num_c}'] = coeficientes[num_c]

                              dataset_filas.append(fila)

                      # reiniciar
                      state = "RESET"

print("\n¡Análisis completado!")
print(f"Total de latidos guardados: {len(dataset_filas)}")

In [ ]:
# 1. Convertir lista en tabla
df = pd.DataFrame(dataset_filas)

# 2. LISTA VALIDAS: Solo Normales y las arritmias V y A
etiquetas_validas = ['N', 'V', 'A']

# 3. Aplico el filtro (nos quedamos solo con las filas que tengan esas etiquetas)
df_filtrado = df[df['Clase_Cardiologo'].isin(etiquetas_validas)]

# 4. Resumen de cuántos hay de cada tipo
print("Resumen de latidos limpios para la Red Neuronal:")
print(df_filtrado['Clase_Cardiologo'].value_counts())

# 5. Guardar el archivo CSV en el entorno de Colab
nombre_archivo = "dataset_hermite_corazon_descendienteL_c5.csv"
df_filtrado.to_csv(nombre_archivo, index=False)

print(f"\n¡Tabla filtrada con éxito! El archivo '{nombre_archivo}' está listo.")
display(df_filtrado.head()) # Muestra las primeras 5 filas para comprobar

# 6. Descarga automática
print("\nIniciando descarga")
files.download(nombre_archivo)

#RED NEURONAL

In [ ]:
# 1. Cargar el dataset en bruto
df = pd.read_csv('dataset_hermite_corazon_descendienteL_c5.csv')

# 3. Convertir a problema binario (0 = Normal, 1 = Arritmia)
def asignar_etiqueta(clase):
    if clase == 'N':
        return 0  # Clase 0: Latido Sano
    else:
        return 1  # Clase 1: Arritmia ('V' o 'A')

df['Etiqueta_Num'] = df['Clase_Cardiologo'].apply(asignar_etiqueta)

# 4. Separar los datos por clase
df_normales = df[df['Etiqueta_Num'] == 0]
df_arritmias = df[df['Etiqueta_Num'] == 1]

num_arritmias = len(df_arritmias)
print(f"Tenemos {len(df_normales)} Normales y {num_arritmias} Arritmias.")

# 5. UNDERSAMPLING: Coger normales al azar para igualar a las arritmias
# random_state=42 asegura que siempre cojamos los mismos latidos aleatorios si repetimos el código
df_normales_reducido = df_normales.sample(n=num_arritmias, random_state=42)

# 6. Juntar ambos grupos y barajar (shuffle)
df_final = pd.concat([df_normales_reducido, df_arritmias])
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n--- DATASET FINAL BALANCEADO ---")
print(df_final['Etiqueta_Num'].value_counts())

#  ver las primeras filas para confirmar que están barajados y listos
display(df_final.head())

In [ ]:
# 1. Separar las Características (X) de las Respuestas (y)
# X: Todas las columnas de coefs
# y: La columna con la solución (0 o 1)

columnas_hermite = ['c0', 'c1', 'c2', 'c3', 'c5']
X = df_final[columnas_hermite]
y = df_final['Etiqueta_Num']

# 2. Hacer la partición 70% / 30% que pide tu tutor
# test_size=0.3 significa 30% para test y 70% para train
# stratify=y asegura que en el examen haya exactamente un 50/50 de arritmias y normales, igual que en el estudio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Total de latidos: {len(df_final)}")
print(f"Latidos para que la red ESTUDIE (Train 70%): {len(X_train)}")
print(f"Latidos para el EXAMEN FINAL (Test 30%): {len(X_test)}")

In [ ]:
# 1. Crear el esqueleto de la Red Neuronal
modelo = Sequential()

# 2. CAPA DE ENTRADA y 1ª CAPA OCULTA
# input_dim=6 porque le vamos a pasar 6 datos (c0, c1, c2, c3, c4, c5)
modelo.add(Dense(16, input_dim=5, activation='relu'))

# 3. 2ª CAPA OCULTA
modelo.add(Dense(8, activation='relu'))

# 4. CAPA DE SALIDA
# 1 sola neurona. 'sigmoid' porque es 0 o 1.
modelo.add(Dense(1, activation='sigmoid'))

# 5. COMPILAR EL MODELO
modelo.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 6. ENTRENAMIENTO
# epochs=50: Le va a dar 50 pasadas de estudio
# validation_data=(X_test, y_test): Al final de cada pasada, hará el examen del 30% para ver si mejora
print("Iniciando el entrenamiento de la Red Neuronal...")
historial = modelo.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

print("\n¡Entrenamiento terminado!")

In [ ]:
# 1. GRÁFICAS DE ENTRENAMIENTO (Accuracy y Loss)
plt.figure(figsize=(12, 5))

# Gráfica de Precisión
plt.subplot(1, 2, 1)
plt.plot(historial.history['accuracy'], label='Train (Estudio)', color='blue')
plt.plot(historial.history['val_accuracy'], label='Test (Examen)', color='orange')
plt.title('Evolución de la Precisión')
plt.xlabel('Épocas')
plt.ylabel('Precisión (0 a 1)')
plt.legend()
plt.grid(True, alpha=0.3)

# Gráfica de Error (Loss)
plt.subplot(1, 2, 2)
plt.plot(historial.history['loss'], label='Train (Estudio)', color='blue')
plt.plot(historial.history['val_loss'], label='Test (Examen)', color='orange')
plt.title('Evolución del Error')
plt.xlabel('Épocas')
plt.ylabel('Error (Loss)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# 2. MATRIZ DE CONFUSIÓN
print("\n--- Generando Matriz de Confusión ---")
y_pred_probabilidades = modelo.predict(X_test)
y_pred = np.round(y_pred_probabilidades > 0.30).astype(int)

# Calculamos la matriz comparando lo que predijo la red (y_pred) con la realidad (y_test)
cm = confusion_matrix(y_test, y_pred)

# La dibujamos
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal (0)', 'Arritmia (1)'])
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title("Matriz de Confusión (Datos de Test)")
plt.show()